## Assignment 2, Part 3: Regression with the Abalone Dataset

### Donita Monroe

### October 25, 2026

This analysis develops and evaluates regression models for predicting the number of rings in abalone using the Kaggle Regression with an Abalone Dataset competition data. The workflow includes data inspection, exploratory data analysis, evaluation of regression assumptions, feature selection, validation, and model comparison.

In [ ]:
pip install kagglehub


In [ ]:
# Import libraries for data handling, Kaggle access, numerical operations, and visualization
import pandas as pd
import kagglehub
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_squared_log_error,
    r2_score
)

import scipy.stats as stats


## Data Acquisition

The competition data were accessed through Kaggle using the `kagglehub` API. The downloaded files include training data, test data, and a sample submission file that defines the required competition submission format.

In [ ]:
# Authenticate access to Kaggle competition data
kagglehub.login()


In [ ]:
# Download and extract the Abalone competition files
path = kagglehub.competition_download(
    "playground-series-s4e4",
    output_dir="./abalone_data"
)

print(path)


In [ ]:
# Verify that the expected competition files were downloaded
import os

os.listdir("./abalone_data")


## Data Loading and Initial Inspection

The training, test, and sample-submission files were loaded into pandas DataFrames. Initial inspection was used to verify dataset dimensions, variable types, missing values, categorical structure, and descriptive statistics before modeling.

In [ ]:
# Load training, test, and submission-template datasets
train = pd.read_csv("./abalone_data/train.csv")
test = pd.read_csv("./abalone_data/test.csv")
sample_sub = pd.read_csv("./abalone_data/sample_submission.csv")


In [ ]:
# Check dataset dimensions and confirm expected competition files
print(train.shape)
print(test.shape)
print(sample_sub.shape)


In [ ]:
train.head()


In [ ]:
sample_sub.head()


In [ ]:
train.info()


In [ ]:
# Confirm whether missing-value treatment is required
train.isna().sum()


In [ ]:
# Examine the distribution of the categorical predictor
train["Sex"].value_counts()


In [ ]:
train.describe()


### Initial Inspection Findings

The training dataset contains 90,615 observations and 10 variables. `Rings` is the target variable, `id` is an identifier, `Sex` is categorical, and the remaining predictors are numeric measurements. No missing values were identified, so missing-value imputation is not required.

The `Sex` variable contains three reasonably represented categories: Infant, Male, and Female. Initial descriptive statistics show that `Rings` ranges from 1 to 29, with a median of 9 and an average of approximately 9.7.

In [ ]:
# Visualize Rings using one bin for each integer value
train["Rings"].hist(
    bins=np.arange(0.5, train["Rings"].max() + 1.5, 1)
)

plt.xlabel("Rings")
plt.ylabel("Frequency")
plt.title("Distribution of Rings")
plt.show()


## Exploratory Data Analysis

### Target Distribution and Outlier Screening

The distribution of the target variable was examined to identify its central tendency, shape, and possible extreme observations. Because `Rings` is discrete, the histogram uses one bin for each integer ring count.

In [ ]:
# Visualize Rings and calculate IQR-based outlier thresholds
train.boxplot(column="Rings")

Q1 = train["Rings"].quantile(0.25)
Q3 = train["Rings"].quantile(0.75)

IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

upper_fence


In [ ]:
# Count observations outside the IQR-based outlier thresholds
lower_fence = 3.5
upper_fence = 15.5

print("Lower outliers:", (train["Rings"] < lower_fence).sum())
print("Upper outliers:", (train["Rings"] > upper_fence).sum())


The much larger number of upper-end observations is consistent with the observed right skew. These values are treated as statistical outliers rather than assumed data errors and are retained for modeling unless later modeling or diagnostic results indicate that they substantially influence predictive performance.

### Target Distribution and Outlier Screening

The `Rings` distribution is positively skewed, with most observations concentrated around the middle values and a longer upper tail. Using the 1.5×IQR rule, values below 3.5 and above 15.5 were flagged as potential outliers. This identified 440 lower-end observations and 5,285 upper-end observations.

The much larger number of upper-end observations is consistent with the observed right-skew. These values are treated as statistical outliers rather than assumed data errors, so they are retained for modeling unless later diagnostics provide evidence that they adversely affect model performance.

### Predictor Relationships

Correlations between the numeric predictors and `Rings` were examined to identify variables with stronger individual linear relationships with the target. Predictor-to-predictor correlations were then examined separately to identify possible redundancy among the explanatory variables.

In [ ]:
# Examine the correlation between numeric variables and the target
train.corr(numeric_only = True)["Rings"].sort_values(ascending = False)


In [ ]:
corr_matrix = train.drop(columns = "id").corr(numeric_only = True)

plt.figure(figsize = (10, 8))
plt.imshow(corr_matrix, aspect = "auto")
plt.colorbar()

plt.xticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90
)
plt.yticks(
    range(len(corr_matrix.columns)),
    corr_matrix.columns
)

plt.title("Correlation Matrix of Numeric Variables")
plt.tight_layout()
plt.show()


### Correlation Findings

`Shell weight` showed the strongest individual correlation with `Rings`, followed by Height, Diameter, Length, and Whole weight. The physical measurements generally showed moderate-to-strong positive relationships with the target.

The heatmap also revealed substantial correlations among several predictors. Length and Diameter were highly correlated with one another, while Whole weight was strongly related to the other weight measurements and to Length and Diameter. These relationships suggest that several predictors contain overlapping information related to overall abalone size and body mass.

Because `id` is an identifier rather than a meaningful characteristic of the abalone, it is excluded from the regression models.

## Multicollinearity Assessment

Variance Inflation Factor (VIF) was used to quantify multicollinearity among the candidate predictors. VIF values near 1 indicate little overlap with the other predictors, values above 5 may warrant further examination, and values above 10 indicate substantial multicollinearity.

The categorical variable `Sex` was dummy encoded, with Female serving as the reference category. The target variable `Rings` and identifier `id` were excluded from the VIF calculation.

In [ ]:
# Create predictor set, excluding the identifier and target
X_vif = train.drop(columns=["id", "Rings"]).copy()

# Convert the categorical Sex variable into numeric dummy variables
X_vif = pd.get_dummies(
    X_vif,
    columns=["Sex"],
    drop_first=True,
    dtype=int
)

# Add an intercept for the VIF calculation
X_vif = sm.add_constant(X_vif)

X_vif.head()


In [ ]:
# Calculate VIF for each predictor
vif_results = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

# Remove the intercept and sort from highest to lowest VIF
vif_results = (
    vif_results[vif_results["Variable"] != "const"]
    .sort_values("VIF", ascending=False)
)

vif_results


### Initial VIF Findings

The initial VIF analysis confirmed substantial multicollinearity among the physical measurements. Whole weight had the highest VIF, followed by Diameter and Length. The remaining weight variables also showed elevated VIF values, while the encoded `Sex` variables showed little evidence of problematic multicollinearity.

Whole weight was removed first because it showed the highest degree of redundancy with the remaining predictors.

In [ ]:
X_vif_2 = X_vif.drop(columns=["Whole weight"])

vif_results_2 = pd.DataFrame({
    "Variable": X_vif_2.columns,
    "VIF": [
        variance_inflation_factor(X_vif_2.values, i)
        for i in range(X_vif_2.shape[1])
    ]
})

vif_results_2 = (
    vif_results_2[vif_results_2["Variable"] != "const"]
    .sort_values("VIF", ascending=False)
)

vif_results_2


In [ ]:
X_vif_3 = X_vif_2.drop(columns=["Length"])

vif_results_3 = pd.DataFrame({
    "Variable": X_vif_3.columns,
    "VIF": [
        variance_inflation_factor(X_vif_3.values, i)
        for i in range(X_vif_3.shape[1])
    ]
})

vif_results_3 = (
    vif_results_3[vif_results_3["Variable"] != "const"]
    .sort_values("VIF", ascending=False)
)

vif_results_3


In [ ]:
# Alternative test: keep Length and remove Diameter
X_vif_3_length = X_vif_2.drop(columns=["Diameter"])

vif_results_3_length = pd.DataFrame({
    "Variable": X_vif_3_length.columns,
    "VIF": [
        variance_inflation_factor(X_vif_3_length.values, i)
        for i in range(X_vif_3_length.shape[1])
    ]
})

vif_results_3_length = (
    vif_results_3_length[
        vif_results_3_length["Variable"] != "const"
    ]
    .sort_values("VIF", ascending=False)
)

vif_results_3_length


### Iterative Feature Selection

Removing Whole weight substantially reduced the VIF values of the remaining weight-related predictors, although severe multicollinearity remained between Length and Diameter.

Removing Length reduced Diameter's VIF from approximately 54 to 10.74. The reverse experiment produced a similar result: removing Diameter reduced Length's VIF from approximately 51 to 10.14. These results indicate that much of the extreme multicollinearity associated with these variables resulted from their strong relationship with one another.

Because either variable could reasonably be retained based on VIF alone, predictive validation was used to determine which feature should remain in Model 1.

### Length Versus Diameter Validation

Two otherwise identical regression models were evaluated using the same 80/20 training-validation split. One retained Length and excluded Diameter, while the other retained Diameter and excluded Length. RMSLE, RMSE, MAE, and $R^2$ were compared so that the final feature-selection decision reflected predictive performance rather than multicollinearity alone.

In [ ]:
# Prepare predictors
X = train.drop(columns=["id", "Rings", "Whole weight"]).copy()

# Encode Sex using Female as the reference category
X = pd.get_dummies(
    X,
    columns=["Sex"],
    drop_first=True,
    dtype=int
)

y = train["Rings"]

# Use one split for both models
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


In [ ]:
# Model A: keep Length, remove Diameter
length_features = [
    col for col in X.columns
    if col != "Diameter"
]

# Model B: keep Diameter, remove Length
diameter_features = [
    col for col in X.columns
    if col != "Length"
]


In [ ]:
model_length = LinearRegression()

model_length.fit(
    X_train[length_features],
    y_train
)

pred_length = model_length.predict(
    X_valid[length_features]
)


model_diameter = LinearRegression()

model_diameter.fit(
    X_train[diameter_features],
    y_train
)

pred_diameter = model_diameter.predict(
    X_valid[diameter_features]
)


In [ ]:
def evaluate_model(y_true, y_pred):
    
    # RMSLE cannot use negative predictions
    y_pred_nonnegative = np.maximum(y_pred, 0)
    
    return {
        "RMSLE": np.sqrt(
            mean_squared_log_error(
                y_true,
                y_pred_nonnegative
            )
        ),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "R2": r2_score(
            y_true,
            y_pred
        )
    }


comparison = pd.DataFrame({
    "Keep Length": evaluate_model(
        y_valid,
        pred_length
    ),
    "Keep Diameter": evaluate_model(
        y_valid,
        pred_diameter
    )
}).T

comparison


The model retaining Diameter performed slightly better across all four validation metrics. Although the differences were small, Diameter produced lower RMSLE, RMSE, and MAE and a slightly higher $R^2$. Diameter was therefore retained and Length was excluded from Model 1.

## Model 1: Multiple Linear Regression

Model 1 uses multiple linear regression as the baseline predictive model. Based on the exploratory analysis and multicollinearity diagnostics, `id`, Whole weight, and Length were excluded. The model retains Diameter, Height, Whole weight.1, Whole weight.2, Shell weight, and the encoded `Sex` variables.

Some elevated multicollinearity remains among the weight-related variables, but these predictors are retained in the baseline model so their predictive contribution and model diagnostics can be evaluated before further refinement.

In [ ]:
# Define predictors for Model 1
model_1_features = [
    "Diameter",
    "Height",
    "Whole weight.1",
    "Whole weight.2",
    "Shell weight",
    "Sex_I",
    "Sex_M"
]

# Fit Model 1
model_1 = LinearRegression()

model_1.fit(
    X_train[model_1_features],
    y_train
)

# Generate validation predictions
pred_1 = model_1.predict(
    X_valid[model_1_features]
)


In [ ]:
model_1_results = evaluate_model(
    y_valid,
    pred_1
)

pd.Series(model_1_results, name="Model 1")


### Model 1 Performance

Model 1 produced an RMSLE of 0.1668, RMSE of 2.04, MAE of 1.40, and $R^2$ of 0.596 on the validation data. The model therefore explains approximately 59.6% of the variation in `Rings`, with predictions differing from observed values by approximately 1.4 rings on average.

These results establish the baseline performance against which subsequent models will be evaluated.

## Model 1 Diagnostic Assessment

Regression diagnostics were examined to determine whether Model 1 adequately satisfies the assumptions underlying linear regression. Residual patterns were evaluated for linearity and constant variance, while a histogram and Q–Q plot were used to examine the distribution of the residuals.

In [ ]:
residuals_1 = y_valid - pred_1

plt.scatter(pred_1, residuals_1, alpha=0.3)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Rings")
plt.ylabel("Residuals")
plt.title("Model 1: Residuals vs. Predicted Values")
plt.show()


In [ ]:
residuals_1 = y_valid - pred_1


In [ ]:
# Examine the distribution of Model 1 residuals
plt.hist(residuals_1, bins=30, edgecolor="black")

plt.axvline(0, linestyle="--")
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Model 1: Distribution of Residuals")
plt.show()


In [ ]:
# Compare residuals with a theoretical normal distribution
stats.probplot(residuals_1, dist="norm", plot=plt)

plt.title("Model 1: Q–Q Plot of Residuals")
plt.show()


### Model 1 Diagnostic Findings

Feature reduction substantially improved the most severe multicollinearity identified during the initial VIF analysis, although some elevated collinearity remained among the weight-related predictors.

The residual-versus-predicted plot showed changing residual spread across portions of the fitted range, suggesting heteroscedasticity. The residual histogram was positively skewed rather than symmetric, and the Q–Q plot showed clear departures from the normal reference line in both tails, particularly the upper tail.

These findings indicate that Model 1 does not fully capture the behavior of extreme `Rings` values and provide justification for evaluating an alternative model specification.

## Model 1 Kaggle Submission

After completing local validation and diagnostic assessment, Model 1 was refit using the complete labeled training dataset. The same preprocessing and finalized seven-predictor feature set were then applied to the competition test data. Predictions were constrained to nonnegative values to satisfy the RMSLE evaluation metric and formatted according to Kaggle's required submission structure.

In [ ]:
# Prepare full training and competition test predictors
X_full = train.drop(columns=["id", "Rings", "Whole weight", "Length"]).copy()
X_test_kaggle = test.drop(columns=["id", "Whole weight", "Length"]).copy()

# Encode Sex using the same reference-category approach used during validation
X_full = pd.get_dummies(
    X_full,
    columns=["Sex"],
    drop_first=True,
    dtype=int
)

X_test_kaggle = pd.get_dummies(
    X_test_kaggle,
    columns=["Sex"],
    drop_first=True,
    dtype=int
)

# Ensure the test data contain the same predictor columns as the training data
X_test_kaggle = X_test_kaggle.reindex(
    columns=X_full.columns,
    fill_value=0
)

y_full = train["Rings"]

X_full.head()


In [ ]:
# Apply the finalized Model 1 feature set
X_full_model_1 = X_full[model_1_features]
X_test_model_1 = X_test_kaggle[model_1_features]

print("Training shape:", X_full_model_1.shape)
print("Kaggle test shape:", X_test_model_1.shape)
print("Features:", X_full_model_1.columns.tolist())


In [ ]:
# Refit Model 1 using the complete training dataset
model_1_final = LinearRegression()

model_1_final.fit(
    X_full_model_1,
    y_full
)


In [ ]:
# Generate predictions for the Kaggle competition test data
kaggle_pred_1 = model_1_final.predict(
    X_test_model_1
)

# RMSLE requires nonnegative predictions
kaggle_pred_1 = np.clip(kaggle_pred_1, 0, None)

kaggle_pred_1[:10]


In [ ]:
# Create the Model 1 Kaggle submission file
submission_1 = sample_sub.copy()

submission_1["Rings"] = kaggle_pred_1

submission_1.head()


In [ ]:
# Confirm submission dimensions and check for invalid predictions
print("Submission shape:", submission_1.shape)
print("Missing predictions:", submission_1["Rings"].isna().sum())
print("Minimum prediction:", submission_1["Rings"].min())
print("Maximum prediction:", submission_1["Rings"].max())


### Submission Validation

The submission contained 60,411 predictions, matching the number of observations in the competition test dataset. No predictions were missing, and all predicted values were nonnegative. These checks confirmed that the file was structurally ready for Kaggle submission.

In [ ]:
# Save Model 1 predictions in Kaggle submission format
submission_1.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv saved successfully.")


In [ ]:
# Verify the saved submission file
pd.read_csv("submission.csv").head()


In [ ]:
import os

print(os.getcwd())
print(os.path.abspath("submission.csv"))


## Model 2: Histogram Gradient Boosting Regression

Model 1 provided a stable linear baseline but showed evidence of heteroscedasticity, non-normal residuals, and systematic error at more extreme `Rings` values. Model 2 therefore uses histogram gradient boosting, a nonlinear tree-based regression method capable of modeling more complex relationships and interactions among predictors.

Unlike multiple linear regression, the tree-based model does not require normally distributed residuals, constant error variance, or low multicollinearity for coefficient interpretation. Therefore, the previously excluded Length and Whole weight predictors are restored for Model 2, while `id` remains excluded because it is an identifier.

Because the competition is evaluated using RMSLE, Model 2 is trained using `log1p(Rings)` as the target. Predictions will subsequently be transformed back to the original `Rings` scale before evaluation.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor


In [ ]:
# Prepare predictors for the nonlinear Model 2
X_model_2 = train.drop(
    columns=["id", "Rings"]
).copy()

# Encode the categorical Sex variable
X_model_2 = pd.get_dummies(
    X_model_2,
    columns=["Sex"],
    drop_first=True,
    dtype=int
)

# Use the same validation observations as Model 1
X2_train = X_model_2.loc[X_train.index]
X2_valid = X_model_2.loc[X_valid.index]

print("Training shape:", X2_train.shape)
print("Validation shape:", X2_valid.shape)
print("Features:", X_model_2.columns.tolist())


In [ ]:
# Align target values with the Model 2 training and validation observations
y2_train = y.loc[X2_train.index]
y2_valid = y.loc[X2_valid.index]

# Transform the target to the log scale used by the RMSLE metric
y2_train_log = np.log1p(y2_train)

# Fit Histogram Gradient Boosting Regression
model_2 = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

model_2.fit(
    X2_train,
    y2_train_log
)

# Transform predictions back to the original Rings scale
pred_2_log = model_2.predict(X2_valid)
pred_2 = np.expm1(pred_2_log)

# Prevent negative predictions before calculating RMSLE
pred_2 = np.clip(pred_2, 0, None)

pred_2[:10]


In [ ]:
# Evaluate Model 2 on the same validation observations used for Model 1
model_2_results = evaluate_model(
    y2_valid,
    pred_2
)

pd.Series(model_2_results, name="Model 2")


In [ ]:
# Compare Model 1 and Model 2 on the same validation observations
model_comparison = pd.DataFrame({
    "Model 1 - Linear Regression": model_1_results,
    "Model 2 - HistGradientBoosting": model_2_results
}).T

model_comparison


### Model Comparison

Model 2 outperformed Model 1 across all four validation metrics. RMSLE decreased from 0.1668 to 0.1519, while RMSE and MAE also declined. In addition, $R^2$ increased from 0.596 to 0.655, indicating that the gradient boosting model explained a larger proportion of variation in `Rings`.

The improvement supports the diagnostic findings from Model 1. The nonlinear gradient boosting model was better able to capture relationships and interactions that were not adequately represented by the multiple linear regression baseline. Model 2 was therefore selected for Kaggle evaluation.

In [ ]:
# Prepare the full Model 2 training target on the log scale
y_full_model_2_log = np.log1p(train["Rings"])

# Prepare Kaggle test predictors using the same preprocessing as Model 2
X_test_model_2 = test.drop(
    columns=["id"]
).copy()

X_test_model_2 = pd.get_dummies(
    X_test_model_2,
    columns=["Sex"],
    drop_first=True,
    dtype=int
)

# Ensure training and test predictors have identical columns and order
X_test_model_2 = X_test_model_2.reindex(
    columns=X_model_2.columns,
    fill_value=0
)

print("Full training shape:", X_model_2.shape)
print("Kaggle test shape:", X_test_model_2.shape)
print("Features match:", X_model_2.columns.tolist() == X_test_model_2.columns.tolist())


In [ ]:
# Refit Model 2 using the complete training dataset
model_2_final = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

model_2_final.fit(
    X_model_2,
    y_full_model_2_log
)

# Generate predictions for the Kaggle test data
kaggle_pred_2_log = model_2_final.predict(
    X_test_model_2
)

# Transform predictions back to the original Rings scale
kaggle_pred_2 = np.expm1(kaggle_pred_2_log)

# Ensure predictions are valid for RMSLE
kaggle_pred_2 = np.clip(
    kaggle_pred_2,
    0,
    None
)

kaggle_pred_2[:10]


In [ ]:
# Create the Model 2 Kaggle submission file
submission_2 = sample_sub.copy()
submission_2["Rings"] = kaggle_pred_2

# Validate the submission structure and predictions
print("Submission shape:", submission_2.shape)
print("Missing predictions:", submission_2["Rings"].isna().sum())
print("Minimum prediction:", submission_2["Rings"].min())
print("Maximum prediction:", submission_2["Rings"].max())

submission_2.head()


In [ ]:
# Save Model 2 predictions in Kaggle submission format
submission_2.to_csv(
    "submission_model_2.csv",
    index=False
)

print("submission_model_2.csv saved successfully.")


In [ ]:
# Verify the saved Model 2 submission file
submission_2_check = pd.read_csv("submission_model_2.csv")

print("Shape:", submission_2_check.shape)
print("Columns:", submission_2_check.columns.tolist())
print("Missing values:", submission_2_check.isna().sum().to_dict())

submission_2_check.head()


In [ ]:
# Summarize the distribution of Model 2 Rings predictions
prediction_summary = submission_2_check["Rings"].describe()

print(prediction_summary)
print("\nMedian:", submission_2_check["Rings"].median())
print("Predictions equal to 0:", (submission_2_check["Rings"] == 0).sum())


In [ ]:
# Visualize the distribution of Model 2 Rings predictions
plt.figure(figsize=(8, 5))

plt.hist(
    submission_2_check["Rings"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("Predicted Rings")
plt.ylabel("Frequency")
plt.title("Distribution of Model 2 Kaggle Predictions")
plt.show()


In [ ]:
# Examine the range and potential extreme Model 2 predictions
plt.figure(figsize=(8, 3))

plt.boxplot(
    submission_2_check["Rings"],
    vert=False
)

plt.xlabel("Predicted Rings")
plt.title("Model 2 Kaggle Prediction Distribution")
plt.show()


In [ ]:
# Compare actual training Rings with Model 2 Kaggle predictions
distribution_comparison = pd.DataFrame({
    "Training Rings": train["Rings"].describe(),
    "Model 2 Predictions": submission_2_check["Rings"].describe()
})

distribution_comparison


### Model 2 Kaggle Results

Model 2 was successfully submitted to Kaggle and received a public RMSLE of **0.14851** and a private RMSLE of **0.14790**. Both scores improved upon the local validation RMSLE of 0.15188 and substantially outperformed the Model 1 Kaggle scores of 0.16502 publicly and 0.16411 privately.

The improvement confirms that histogram gradient boosting captured predictive relationships that were not adequately represented by the multiple linear regression baseline. The consistency between the validation and Kaggle results also supports the reliability of the model comparison and indicates that Model 2 generalized well to unseen observations.

### Final Model Comparison

Across both local validation and Kaggle evaluation, Model 2 consistently outperformed Model 1. Multiple linear regression provided a useful and interpretable baseline, but diagnostic analysis revealed heteroscedasticity, non-normal residuals, remaining multicollinearity, and difficulty capturing extreme `Rings` values. Histogram gradient boosting addressed the primary predictive limitation by modeling nonlinear relationships and interactions among the predictors without relying on the distributional assumptions of ordinary least squares.

Based on its lower validation and Kaggle RMSLE values, Model 2 was selected as the stronger predictive model.